# TO Number Process

## Step 1: LTAP

In [ ]:
import os
import time
import ctypes
import win32com.client


# =========================================================
# Helper Functions
# =========================================================

def msgbox(message, title="SAP Python Script"):
    ctypes.windll.user32.MessageBoxW(0, message, title, 0)


def normalize_key(value):
    if value is None:
        txt = ""
    else:
        txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")
    txt = txt.replace("'", "")
    txt = txt.replace(",", "")

    if txt.endswith(".0"):
        txt = txt[:-2]

    if txt.endswith(".00"):
        txt = txt[:-3]

    if txt.endswith("."):
        txt = txt[:-1]

    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]

    return txt


def clean_numeric_for_sap(value):
    """
    Cleans Excel values before pasting into SAP numeric fields.
    Example:
    123456789.0 -> 123456789
    123456789.  -> 123456789
    """
    if value is None:
        return ""

    txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")
    txt = txt.replace(",", "")
    txt = txt.replace("'", "")

    if txt.endswith(".0"):
        txt = txt[:-2]

    if txt.endswith(".00"):
        txt = txt[:-3]

    if txt.endswith("."):
        txt = txt[:-1]

    return txt


def get_cell_text(cell):
    try:
        txt = str(cell.Text).strip()
    except Exception:
        txt = ""

    if txt == "" or "#" in txt:
        try:
            txt = str(cell.Value).strip()
        except Exception:
            txt = ""

    return txt


def get_sap_session():
    sap_gui_auto = win32com.client.GetObject("SAPGUI")
    application = sap_gui_auto.GetScriptingEngine
    connection = application.Children(0)
    session = connection.Children(0)
    return sap_gui_auto, application, connection, session


def get_visible_row_count(grid):
    try:
        visible_count = int(grid.VisibleRowCount)
        if visible_count > 0:
            return visible_count
    except Exception:
        pass

    return 20


def scroll_sap_grid_to_row(grid, row_number):
    try:
        grid.firstVisibleRow = row_number
        return True
    except Exception:
        pass

    try:
        grid.VerticalScrollbar.Position = row_number
        return True
    except Exception:
        pass

    return False


def safe_get_sap_cell(grid, row_number, column_name):
    for attempt in range(3):
        try:
            return grid.GetCellValue(row_number, column_name)
        except Exception:
            scroll_sap_grid_to_row(grid, row_number)
            time.sleep(0.2)

    return ""


# =========================================================
# Main Script
# =========================================================

excel_file_path = r"C:\Users\E10878194\OneDrive - RTX\SAP Script Testing\GTF SS Database - TEST.xlsx"

xl_up = -4162

# =========================================================
# Column Mapping After Swapping Column B with Column K
# =========================================================

to_number_col = 11         # Column K
unloading_point_col = 13   # Column M

xl_app = None
xl_book = None
xl_sheet = None
temp_book = None
temp_sheet = None

sap_gui_auto = None
application = None
connection = None
session = None
grid = None

try:
    # =========================================================
    # Open Excel and copy valid TO Numbers from Column K
    # =========================================================

    if not os.path.exists(excel_file_path):
        msgbox(f"Excel file not found. Please check this path:\n{excel_file_path}")
        raise SystemExit

    xl_app = win32com.client.Dispatch("Excel.Application")
    xl_app.Visible = False
    xl_app.DisplayAlerts = False

    xl_book = xl_app.Workbooks.Open(excel_file_path, 0, False)

    if xl_book.ReadOnly:
        msgbox(
            "Excel file opened as Read-Only. Please close the file or check "
            f"OneDrive/SharePoint permissions:\n{excel_file_path}"
        )
        raise SystemExit

    xl_sheet = xl_book.Worksheets(1)

    # TO Number is now in Column K
    last_row = xl_sheet.Cells(xl_sheet.Rows.Count, to_number_col).End(xl_up).Row

    if last_row < 2:
        msgbox("No TO Number found in Column K.")
        raise SystemExit

    temp_book = xl_app.Workbooks.Add()
    temp_sheet = temp_book.Worksheets(1)
    temp_sheet.Columns("A").NumberFormat = "@"

    temp_row = 1

    for i in range(2, last_row + 1):
        to_number = clean_numeric_for_sap(get_cell_text(xl_sheet.Cells(i, to_number_col)))

        if to_number != "" and to_number.upper() != "NOT FOUND" and to_number.upper() != "NOTFOUND":
            temp_sheet.Cells(temp_row, 1).Value = to_number
            temp_row += 1

    if temp_row == 1:
        msgbox("No valid TO Number found in Column K.")
        raise SystemExit

    temp_sheet.Range(f"A1:A{temp_row - 1}").Copy()

    # =========================================================
    # Connect to SAP
    # =========================================================

    sap_gui_auto, application, connection, session = get_sap_session()

    # =========================================================
    # Go into ZTBV and LTAP
    # =========================================================

    session.findById("wnd[0]").maximize()

    session.findById("wnd[0]/tbar[0]/okcd").Text = "/nZTBV"
    session.findById("wnd[0]").sendVKey(0)

    time.sleep(2)

    session.findById("wnd[0]/usr/txtD_WERKS").Text = "esa1"
    session.findById("wnd[0]/usr/ctxtD_TAB").Text = "ltap"
    session.findById("wnd[0]/usr/ctxtD_TAB").SetFocus()
    session.findById("wnd[0]/usr/ctxtD_TAB").caretPosition = 4

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(1.5)

    # =========================================================
    # Paste TO Numbers into Transfer Order Number field S3
    # =========================================================

    session.findById("wnd[0]/usr/btn%_S3_%_APP_%-VALU_PUSH").press()

    time.sleep(1)

    # Added: clear values kept from a previous run -- SAP retains the
    # multi-select dialog's contents within a session, and leftovers would
    # silently join this run's filter. Button 16 = Delete Entire Selection
    # (absent on some SAP GUI versions, hence the try).
    try:
        session.findById("wnd[1]/tbar[0]/btn[16]").press()
        time.sleep(0.5)
    except Exception:
        pass

    session.findById("wnd[1]/tbar[0]/btn[24]").press()

    time.sleep(1)

    session.findById("wnd[1]/tbar[0]/btn[8]").press()

    time.sleep(1)

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(3)

    xl_app.CutCopyMode = False

    if temp_book is not None:
        temp_book.Close(False)
        temp_book = None

    # =========================================================
    # Extract Unloading Point from SAP LTAP table
    # =========================================================

    grid = session.findById("wnd[0]/shellcont/shell")

    unload_map = {}

    row_count = int(grid.RowCount)
    visible_row_count = get_visible_row_count(grid)

    for start_row in range(0, row_count, visible_row_count):
        scroll_sap_grid_to_row(grid, start_row)
        time.sleep(0.3)

        end_row = min(start_row + visible_row_count, row_count)

        for sap_row in range(start_row, end_row):
            scroll_sap_grid_to_row(grid, sap_row)

            sap_to_number = normalize_key(safe_get_sap_cell(grid, sap_row, "TANUM"))
            unloading_point = safe_get_sap_cell(grid, sap_row, "ABLAD")

            if sap_to_number != "" and sap_to_number not in unload_map:
                unload_map[sap_to_number] = unloading_point

    # =========================================================
    # Paste matched Unloading Point into Excel Column M
    # =========================================================

    xl_sheet.Range(f"M2:M{xl_sheet.Rows.Count}").ClearContents()
    xl_sheet.Range("M:M").NumberFormat = "@"

    matched_count = 0
    not_matched_count = 0

    for i in range(2, last_row + 1):
        excel_to_number = normalize_key(get_cell_text(xl_sheet.Cells(i, to_number_col)))

        if excel_to_number in unload_map:
            xl_sheet.Cells(i, unloading_point_col).Value = unload_map[excel_to_number]  # Column M
            matched_count += 1
        else:
            xl_sheet.Cells(i, unloading_point_col).Value = ""

            if excel_to_number != "":
                not_matched_count += 1

    xl_sheet.Columns("M:M").AutoFit()

    xl_book.Save()

    msgbox(
        "Completed.\n\n"
        f"SAP LTAP rows detected: {row_count}\n"
        f"Matched TO Numbers: {matched_count}\n"
        f"Not matched TO Numbers: {not_matched_count}\n\n"
        "TO Numbers were read from Column K.\n"
        "Unloading Point data has been pasted into Column M and matched by TO Number."
    )

except SystemExit:
    pass

except Exception as e:
    msgbox(f"Error occurred:\n{str(e)}")

finally:
    try:
        if temp_book is not None:
            temp_book.Close(False)
    except Exception:
        pass

    try:
        if xl_book is not None:
            xl_book.Close(True)
    except Exception:
        pass

    try:
        if xl_app is not None:
            xl_app.Quit()
    except Exception:
        pass

    unload_map = None
    grid = None
    temp_sheet = None
    temp_book = None
    xl_sheet = None
    xl_book = None
    xl_app = None
    session = None
    connection = None
    application = None
    sap_gui_auto = None

## Step 2: Z50CFG_ENG_CRNT

In [ ]:
import os
import time
import ctypes
import win32com.client


# =========================================================
# Helper Functions
# =========================================================

def msgbox(message, title="SAP Python Script"):
    ctypes.windll.user32.MessageBoxW(0, message, title, 0)


def normalize_key(value):
    if value is None:
        txt = ""
    else:
        txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")
    txt = txt.replace("'", "")
    txt = txt.replace(",", "")

    if txt.endswith(".0"):
        txt = txt[:-2]

    if txt.endswith(".00"):
        txt = txt[:-3]

    if txt.endswith("."):
        txt = txt[:-1]

    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]

    return txt


def clean_numeric_for_sap(value):
    """
    Cleans Excel values before pasting into SAP numeric fields.
    Example:
    485247849.0  -> 485247849
    485247849.   -> 485247849
    485,247,849  -> 485247849
    """
    if value is None:
        return ""

    txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")
    txt = txt.replace(",", "")
    txt = txt.replace("'", "")

    if txt.endswith(".0"):
        txt = txt[:-2]

    if txt.endswith(".00"):
        txt = txt[:-3]

    if txt.endswith("."):
        txt = txt[:-1]

    return txt


def get_cell_text(cell):
    try:
        txt = str(cell.Text).strip()
    except Exception:
        txt = ""

    if txt == "" or "#" in txt:
        try:
            txt = str(cell.Value).strip()
        except Exception:
            txt = ""

    return txt


def get_sap_session():
    sap_gui_auto = win32com.client.GetObject("SAPGUI")
    application = sap_gui_auto.GetScriptingEngine
    connection = application.Children(0)
    session = connection.Children(0)
    return sap_gui_auto, application, connection, session


def get_visible_row_count(grid):
    try:
        visible_count = int(grid.VisibleRowCount)
        if visible_count > 0:
            return visible_count
    except Exception:
        pass

    return 20


def scroll_sap_grid_to_row(grid, row_number):
    try:
        grid.firstVisibleRow = row_number
        return True
    except Exception:
        pass

    try:
        grid.VerticalScrollbar.Position = row_number
        return True
    except Exception:
        pass

    return False


def safe_get_sap_cell(grid, row_number, column_name):
    for attempt in range(3):
        try:
            return grid.GetCellValue(row_number, column_name)
        except Exception:
            scroll_sap_grid_to_row(grid, row_number)
            time.sleep(0.2)

    return ""


def get_or_create_sheet(workbook, sheet_name):
    for idx in range(1, workbook.Worksheets.Count + 1):
        ws = workbook.Worksheets(idx)
        if ws.Name == sheet_name:
            ws.Cells.Clear()
            return ws

    ws = workbook.Worksheets.Add()
    ws.Name = sheet_name
    return ws


# =========================================================
# Main Script
# =========================================================

excel_file_path = r"C:\Users\E10878194\OneDrive - RTX\SAP Script Testing\GTF SS Database - TEST.xlsx"

reservation_button_id = "wnd[0]/usr/btn%_S15_%_APP_%-VALU_PUSH"

xl_up = -4162

# =========================================================
# Column Mapping After Column B and Column K Swap
# =========================================================

qmnum_col = 1             # Column A
objnr_col = 3             # Column C
disp_mat_col = 4          # Column D
disp_qty_col = 5          # Column E

reservation_col = 14      # Column N
item_col = 15             # Column O
match_key_col = 16        # Column P

xl_app = None
xl_book = None
xl_sheet = None
temp_book = None
temp_sheet = None
debug_sheet = None

sap_gui_auto = None
application = None
connection = None
session = None
grid = None

try:
    # =========================================================
    # Open Excel
    # =========================================================

    if not os.path.exists(excel_file_path):
        msgbox(f"Excel file not found. Please check this path:\n{excel_file_path}")
        raise SystemExit

    xl_app = win32com.client.Dispatch("Excel.Application")
    xl_app.Visible = False
    xl_app.DisplayAlerts = False

    xl_book = xl_app.Workbooks.Open(excel_file_path, 0, False)

    if xl_book.ReadOnly:
        msgbox(
            "Excel file opened as Read-Only. Please close the file or check "
            f"OneDrive/SharePoint permissions:\n{excel_file_path}"
        )
        raise SystemExit

    xl_sheet = xl_book.Worksheets(1)

    last_row = xl_sheet.Cells(xl_sheet.Rows.Count, reservation_col).End(xl_up).Row

    if last_row < 2:
        msgbox("No Number of Reservation/Depend found in Column N.")
        raise SystemExit

    # =========================================================
    # Copy only Number of Reservation/Depend from Column N
    # Clean values before pasting into SAP
    # =========================================================

    temp_book = xl_app.Workbooks.Add()
    temp_sheet = temp_book.Worksheets(1)
    temp_sheet.Columns("A").NumberFormat = "@"

    temp_row = 1

    for i in range(2, last_row + 1):
        raw_value = xl_sheet.Cells(i, reservation_col).Value  # Column N

        reservation_no = clean_numeric_for_sap(raw_value)

        if reservation_no != "" and reservation_no.upper() != "NOTFOUND" and reservation_no.upper() != "NOT FOUND":
            temp_sheet.Cells(temp_row, 1).Value = reservation_no
            temp_row += 1

    if temp_row == 1:
        msgbox("No valid Number of Reservation/Depend found in Column N.")
        raise SystemExit

    temp_sheet.Range(f"A1:A{temp_row - 1}").Copy()

    # =========================================================
    # Connect to SAP
    # =========================================================

    sap_gui_auto, application, connection, session = get_sap_session()

    # =========================================================
    # Open ZTBV and load Z50CFG_ENG_CRNT
    # =========================================================

    session.findById("wnd[0]").maximize()
    session.findById("wnd[0]/tbar[0]/okcd").Text = "/nZTBV"
    session.findById("wnd[0]").sendVKey(0)

    time.sleep(2)

    session.findById("wnd[0]/usr/txtD_WERKS").Text = "ESA1"
    session.findById("wnd[0]/usr/ctxtD_TAB").Text = "Z50CFG_ENG_CRNT"
    session.findById("wnd[0]/usr/ctxtD_TAB").SetFocus()
    session.findById("wnd[0]/usr/ctxtD_TAB").caretPosition = 15

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(2)

    # =========================================================
    # Paste Number of Reservation/Depend into SAP S15 only
    # =========================================================

    session.findById(reservation_button_id).press()

    time.sleep(1)

    # Added: clear values kept from a previous run -- SAP retains the
    # multi-select dialog's contents within a session, and leftovers would
    # silently join this run's filter. Button 16 = Delete Entire Selection
    # (absent on some SAP GUI versions, hence the try).
    try:
        session.findById("wnd[1]/tbar[0]/btn[16]").press()
        time.sleep(0.5)
    except Exception:
        pass

    session.findById("wnd[1]/tbar[0]/btn[24]").press()

    time.sleep(1)

    session.findById("wnd[1]/tbar[0]/btn[8]").press()

    time.sleep(1)

    # =========================================================
    # Execute Search
    # =========================================================

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(3)

    xl_app.CutCopyMode = False

    if temp_book is not None:
        temp_book.Close(False)
        temp_book = None

    # =========================================================
    # Create / Clear SAP Debug Sheet
    # =========================================================

    debug_sheet = get_or_create_sheet(xl_book, "SAP_Debug_Table")

    debug_sheet.Range("A1").Value = "SAP Row"
    debug_sheet.Range("B1").Value = "SAP QMNUM Raw"
    debug_sheet.Range("C1").Value = "SAP RSNUM Raw"
    debug_sheet.Range("D1").Value = "SAP RSPOS Raw"
    debug_sheet.Range("E1").Value = "SAP Match Key Used"
    debug_sheet.Range("F1").Value = "OBJNR"
    debug_sheet.Range("G1").Value = "DISP_MATNR"
    debug_sheet.Range("H1").Value = "DISP_QTY"

    debug_sheet.Range("A:H").NumberFormat = "@"

    # =========================================================
    # Extract SAP data
    # Scroll through entire SAP table and paste into SAP_Debug_Table
    # =========================================================

    grid = session.findById("wnd[0]/shellcont/shell")

    sap_map = {}

    row_count = int(grid.RowCount)
    visible_row_count = get_visible_row_count(grid)

    debug_row = 2
    failed_read_count = 0

    for start_row in range(0, row_count, visible_row_count):

        scroll_sap_grid_to_row(grid, start_row)
        time.sleep(0.3)

        end_row = min(start_row + visible_row_count, row_count)

        for sap_row in range(start_row, end_row):

            try:
                scroll_sap_grid_to_row(grid, sap_row)
                time.sleep(0.05)

                qmnum = safe_get_sap_cell(grid, sap_row, "QMNUM")
                sap_reservation_no_raw = safe_get_sap_cell(grid, sap_row, "RSNUM")
                sap_item_no_raw = safe_get_sap_cell(grid, sap_row, "RSPOS")

                sap_reservation_no = normalize_key(sap_reservation_no_raw)
                sap_item_no = normalize_key(sap_item_no_raw)

                sap_key = f"{sap_reservation_no}|{sap_item_no}"

                obj_no = safe_get_sap_cell(grid, sap_row, "OBJNR")
                disp_mat = safe_get_sap_cell(grid, sap_row, "DISP_MATNR")
                disp_qty = safe_get_sap_cell(grid, sap_row, "DISP_QTY")

                debug_sheet.Cells(debug_row, 1).Value = str(sap_row + 1)
                debug_sheet.Cells(debug_row, 2).Value = qmnum
                debug_sheet.Cells(debug_row, 3).Value = sap_reservation_no_raw
                debug_sheet.Cells(debug_row, 4).Value = sap_item_no_raw
                debug_sheet.Cells(debug_row, 5).Value = sap_key
                debug_sheet.Cells(debug_row, 6).Value = obj_no
                debug_sheet.Cells(debug_row, 7).Value = disp_mat
                debug_sheet.Cells(debug_row, 8).Value = disp_qty

                debug_row += 1

                if sap_reservation_no != "" and sap_item_no != "":
                    if sap_key not in sap_map:
                        sap_map[sap_key] = {
                            "QMNUM": qmnum,
                            "OBJNR": obj_no,
                            "DISP_MATNR": disp_mat,
                            "DISP_QTY": disp_qty
                        }

            except Exception as row_error:
                failed_read_count += 1

                debug_sheet.Cells(debug_row, 1).Value = str(sap_row + 1)
                debug_sheet.Cells(debug_row, 2).Value = "READ ERROR"
                debug_sheet.Cells(debug_row, 3).Value = str(row_error)
                debug_row += 1

    # =========================================================
    # Create Excel Match Key in Column P
    # Paste matched results into A, C, D, E
    # =========================================================

    # Column B and Column K are intentionally left untouched.
    xl_sheet.Range(f"C2:E{xl_sheet.Rows.Count}").ClearContents()

    xl_sheet.Range("A:A").NumberFormat = "@"
    xl_sheet.Range("C:E").NumberFormat = "@"

    xl_sheet.Cells(1, match_key_col).Value = "Excel Match Key Used"  # Column P
    xl_sheet.Columns(match_key_col).NumberFormat = "@"

    matched_count = 0
    not_matched_count = 0

    for i in range(2, last_row + 1):
        reservation_no = normalize_key(get_cell_text(xl_sheet.Cells(i, reservation_col)))  # Column N
        item_no = normalize_key(get_cell_text(xl_sheet.Cells(i, item_col)))                # Column O

        excel_key = f"{reservation_no}|{item_no}"

        xl_sheet.Cells(i, match_key_col).Value = excel_key  # Column P

        if excel_key in sap_map:
            xl_sheet.Cells(i, qmnum_col).Value = sap_map[excel_key]["QMNUM"]          # Column A
            xl_sheet.Cells(i, objnr_col).Value = sap_map[excel_key]["OBJNR"]          # Column C
            xl_sheet.Cells(i, disp_mat_col).Value = sap_map[excel_key]["DISP_MATNR"]  # Column D
            xl_sheet.Cells(i, disp_qty_col).Value = sap_map[excel_key]["DISP_QTY"]    # Column E

            matched_count += 1
            
        else:
            # Do not touch Column A if there is no match
            xl_sheet.Cells(i, objnr_col).Value = ""
            xl_sheet.Cells(i, disp_mat_col).Value = ""
            xl_sheet.Cells(i, disp_qty_col).Value = ""
        
            if reservation_no != "" and item_no != "":
                not_matched_count += 1

    debug_sheet.Columns("A:H").AutoFit()
    xl_sheet.Columns("A:E").AutoFit()
    xl_sheet.Columns("P:P").AutoFit()

    xl_book.Save()

    msgbox(
        "Completed.\n\n"
        f"SAP Z50CFG_ENG_CRNT rows detected: {row_count}\n"
        f"SAP visible rows per screen: {visible_row_count}\n"
        f"SAP rows copied to SAP_Debug_Table: {debug_row - 2}\n"
        f"Failed SAP row reads: {failed_read_count}\n"
        f"Matched Reservation + Item rows: {matched_count}\n"
        f"Not matched Reservation + Item rows: {not_matched_count}\n\n"
        "Reservation Numbers were read from Column N.\n"
        "Item Numbers were read from Column O.\n\n"
        "QMNUM has been pasted into Column A.\n"
        "OBJNR, DISP_MATNR, and DISP_QTY have been pasted into Columns C, D, and E.\n\n"
        "Column B and Column K were not touched by this script.\n\n"
        "Check Column P in Sheet1 against Column E in SAP_Debug_Table."
    )

except SystemExit:
    pass

except Exception as e:
    msgbox(f"Error occurred:\n{str(e)}")

finally:
    # =========================================================
    # Close Excel and clean up
    # =========================================================

    try:
        if temp_book is not None:
            temp_book.Close(False)
    except Exception:
        pass

    try:
        if xl_book is not None:
            xl_book.Close(True)
    except Exception:
        pass

    try:
        if xl_app is not None:
            xl_app.Quit()
    except Exception:
        pass

    sap_map = None
    grid = None
    debug_sheet = None
    temp_sheet = None
    temp_book = None
    xl_sheet = None
    xl_book = None
    xl_app = None
    session = None
    connection = None
    application = None
    sap_gui_auto = None

## Step 3: Z50CFG ENG VALD

In [ ]:
import os
import time
import ctypes
import win32com.client


# =========================================================
# Helper Functions
# =========================================================

def msgbox(message, title="SAP Python Script"):
    ctypes.windll.user32.MessageBoxW(0, message, title, 0)


def normalize_key(value):
    if value is None:
        txt = ""
    else:
        txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")

    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]

    return txt


def get_cell_text(cell):
    try:
        txt = str(cell.Text).strip()
    except Exception:
        txt = ""

    if txt == "" or "#" in txt:
        try:
            txt = str(cell.Value).strip()
        except Exception:
            txt = ""

    return txt


def get_sap_session():
    sap_gui_auto = win32com.client.GetObject("SAPGUI")
    application = sap_gui_auto.GetScriptingEngine
    connection = application.Children(0)
    session = connection.Children(0)
    return sap_gui_auto, application, connection, session


def get_visible_row_count(grid):
    try:
        visible_count = int(grid.VisibleRowCount)
        if visible_count > 0:
            return visible_count
    except Exception:
        pass

    return 20


def scroll_sap_grid_to_row(grid, row_number):
    try:
        grid.firstVisibleRow = row_number
        return True
    except Exception:
        pass

    try:
        grid.VerticalScrollbar.Position = row_number
        return True
    except Exception:
        pass

    return False


def safe_get_sap_cell(grid, row_number, column_name):
    for attempt in range(3):
        try:
            return grid.GetCellValue(row_number, column_name)
        except Exception:
            scroll_sap_grid_to_row(grid, row_number)
            time.sleep(0.2)

    return ""


# =========================================================
# Main Script
# =========================================================

excel_file_path = r"C:\Users\E10878194\OneDrive - RTX\SAP Script Testing\GTF SS Database - TEST.xlsx"

xl_up = -4162

# =========================================================
# Column Mapping After Column B and Column K Swap
# =========================================================

object_no_col = 3       # Column C

section_col = 6         # Column F
module_col = 7          # Column G
description_col = 8     # Column H
sales_doc_col = 9       # Column I
lid_col = 10            # Column J

xl_app = None
xl_book = None
xl_sheet = None
temp_book = None
temp_sheet = None

sap_gui_auto = None
application = None
connection = None
session = None
grid = None

try:
    # =========================================================
    # Open Excel and copy valid Object Numbers from Column C
    # =========================================================

    if not os.path.exists(excel_file_path):
        msgbox(f"Excel file not found. Please check this path:\n{excel_file_path}")
        raise SystemExit

    xl_app = win32com.client.Dispatch("Excel.Application")
    xl_app.Visible = False
    xl_app.DisplayAlerts = False

    xl_book = xl_app.Workbooks.Open(excel_file_path, 0, False)

    if xl_book.ReadOnly:
        msgbox(
            "Excel file opened as Read-Only. Please check OneDrive/SharePoint sync permissions:\n"
            f"{excel_file_path}"
        )
        raise SystemExit

    xl_sheet = xl_book.Worksheets(1)

    last_row = xl_sheet.Cells(xl_sheet.Rows.Count, object_no_col).End(xl_up).Row

    if last_row < 2:
        msgbox("No Object Number found in Column C.")
        raise SystemExit

    temp_book = xl_app.Workbooks.Add()
    temp_sheet = temp_book.Worksheets(1)
    temp_sheet.Columns("A").NumberFormat = "@"

    temp_row = 1

    for i in range(2, last_row + 1):
        object_no = get_cell_text(xl_sheet.Cells(i, object_no_col))  # Column C

        if object_no != "" and object_no.upper() != "NOT FOUND" and object_no.upper() != "NOTFOUND":
            temp_sheet.Cells(temp_row, 1).Value = object_no
            temp_row += 1

    if temp_row == 1:
        msgbox("No valid Object Number found in Column C.")
        raise SystemExit

    temp_sheet.Range(f"A1:A{temp_row - 1}").Copy()

    # =========================================================
    # Connect to SAP
    # =========================================================

    sap_gui_auto, application, connection, session = get_sap_session()

    # =========================================================
    # Open ZTBV and load Z50CFG_ENG_VALD
    # =========================================================

    session.findById("wnd[0]").maximize()
    session.findById("wnd[0]/tbar[0]/okcd").Text = "/nZTBV"
    session.findById("wnd[0]").sendVKey(0)

    time.sleep(2)

    session.findById("wnd[0]/usr/txtD_WERKS").Text = "ESA1"
    session.findById("wnd[0]/usr/ctxtD_TAB").Text = "Z50CFG_ENG_VALD"
    session.findById("wnd[0]/usr/ctxtD_TAB").SetFocus()
    session.findById("wnd[0]/usr/ctxtD_TAB").caretPosition = 15

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(1)

    # =========================================================
    # Paste Object Numbers into S2 and execute
    # =========================================================

    session.findById("wnd[0]/usr/btn%_S2_%_APP_%-VALU_PUSH").press()

    time.sleep(1)

    # Added: clear values kept from a previous run -- SAP retains the
    # multi-select dialog's contents within a session, and leftovers would
    # silently join this run's filter. Button 16 = Delete Entire Selection
    # (absent on some SAP GUI versions, hence the try).
    try:
        session.findById("wnd[1]/tbar[0]/btn[16]").press()
        time.sleep(0.5)
    except Exception:
        pass

    session.findById("wnd[1]/tbar[0]/btn[24]").press()

    time.sleep(1)

    session.findById("wnd[1]/tbar[0]/btn[8]").press()

    time.sleep(1)

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(3)

    xl_app.CutCopyMode = False

    if temp_book is not None:
        temp_book.Close(False)
        temp_book = None

    # =========================================================
    # Extract SAP data and match by Object Number
    # =========================================================

    grid = session.findById("wnd[0]/shellcont/shell")

    sap_data_map = {}

    row_count = int(grid.RowCount)
    visible_row_count = get_visible_row_count(grid)

    failed_read_count = 0

    for start_row in range(0, row_count, visible_row_count):
        scroll_sap_grid_to_row(grid, start_row)
        time.sleep(0.3)

        end_row = min(start_row + visible_row_count, row_count)

        for sap_row in range(start_row, end_row):
            try:
                scroll_sap_grid_to_row(grid, sap_row)

                sap_obj_no = normalize_key(safe_get_sap_cell(grid, sap_row, "OBJNR"))

                if sap_obj_no != "" and sap_obj_no not in sap_data_map:
                    sap_data_map[sap_obj_no] = {
                        "Z_SECTION": safe_get_sap_cell(grid, sap_row, "Z_SECTION"),
                        "Z_MODULE": safe_get_sap_cell(grid, sap_row, "Z_MODULE"),
                        "DESCRIPT": safe_get_sap_cell(grid, sap_row, "DESCRIPT"),
                        "SALES_ORDER": safe_get_sap_cell(grid, sap_row, "SALES_ORDER"),
                    }

            except Exception:
                failed_read_count += 1

    # =========================================================
    # Paste matched results into Excel Columns F to I
    # =========================================================

    xl_sheet.Range(f"F2:I{xl_sheet.Rows.Count}").ClearContents()
    xl_sheet.Range("F:I").NumberFormat = "@"

    matched_count = 0
    not_matched_count = 0

    for i in range(2, last_row + 1):
        excel_obj_no = normalize_key(get_cell_text(xl_sheet.Cells(i, object_no_col)))  # Column C

        if excel_obj_no in sap_data_map:
            xl_sheet.Cells(i, section_col).Value = sap_data_map[excel_obj_no]["Z_SECTION"]          # Column F
            xl_sheet.Cells(i, module_col).Value = sap_data_map[excel_obj_no]["Z_MODULE"]            # Column G
            xl_sheet.Cells(i, description_col).Value = sap_data_map[excel_obj_no]["DESCRIPT"]       # Column H
            xl_sheet.Cells(i, sales_doc_col).Value = sap_data_map[excel_obj_no]["SALES_ORDER"]      # Column I

            matched_count += 1

        else:
            xl_sheet.Cells(i, section_col).Value = ""
            xl_sheet.Cells(i, module_col).Value = ""
            xl_sheet.Cells(i, description_col).Value = ""
            xl_sheet.Cells(i, sales_doc_col).Value = ""
            xl_sheet.Cells(i, lid_col).Value = ""

            if excel_obj_no != "":
                not_matched_count += 1

    xl_sheet.Columns("F:I").AutoFit()

    xl_book.Save()

    msgbox(
        "Completed.\n\n"
        f"SAP Z50CFG_ENG_VALD rows detected: {row_count}\n"
        f"SAP visible rows per screen: {visible_row_count}\n"
        f"Failed SAP row reads: {failed_read_count}\n"
        f"Matched Object Numbers: {matched_count}\n"
        f"Not matched Object Numbers: {not_matched_count}\n\n"
        "Object Numbers were read from Column C.\n"
        "Section, Module, Description, Sales Doc. have been pasted into Columns F to I.\n\n"
        "Column B and Column K were not touched by this script."
    )

except SystemExit:
    pass

except Exception as e:
    msgbox(f"Error occurred:\n{str(e)}")

finally:
    try:
        if temp_book is not None:
            temp_book.Close(False)
    except Exception:
        pass

    try:
        if xl_book is not None:
            xl_book.Close(True)
    except Exception:
        pass

    try:
        if xl_app is not None:
            xl_app.Quit()
    except Exception:
        pass

    sap_data_map = None
    grid = None
    temp_sheet = None
    temp_book = None
    xl_sheet = None
    xl_book = None
    xl_app = None
    session = None
    connection = None
    application = None
    sap_gui_auto = None

# Notification Number Process

## Step 1: Z50CFG_ENG_CRNT

In [ ]:
import os
import time
import ctypes
from decimal import Decimal, InvalidOperation
import win32com.client


# =========================================================
# Helper Functions
# =========================================================

def msgbox(message, title="SAP Python Script"):
    ctypes.windll.user32.MessageBoxW(0, message, title, 0)


def normalize_key(value):
    if value is None:
        txt = ""
    else:
        txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")
    txt = txt.replace("'", "")
    txt = txt.replace(",", "")

    if "E" in txt.upper():
        try:
            txt = format(Decimal(txt), "f")
        except InvalidOperation:
            pass

    if "." in txt:
        left_part, right_part = txt.split(".", 1)
        if right_part == "" or set(right_part) <= {"0"}:
            txt = left_part

    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]

    return txt


def clean_numeric_for_sap(value):
    """
    Cleans Excel values before pasting into SAP numeric fields.
    Example:
    1001234567.0  -> 1001234567
    1001234567.   -> 1001234567
    1.001234567E9 -> 1001234567
    """
    if value is None:
        return ""

    txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")
    txt = txt.replace(",", "")
    txt = txt.replace("'", "")

    if "E" in txt.upper():
        try:
            txt = format(Decimal(txt), "f")
        except InvalidOperation:
            pass

    if "." in txt:
        left_part, right_part = txt.split(".", 1)
        if right_part == "" or set(right_part) <= {"0"}:
            txt = left_part

    return txt


def get_cell_text(cell):
    try:
        txt = str(cell.Text).strip()
    except Exception:
        txt = ""

    if txt == "" or "#" in txt:
        try:
            txt = str(cell.Value).strip()
        except Exception:
            txt = ""

    return txt


def get_sap_session():
    sap_gui_auto = win32com.client.GetObject("SAPGUI")
    application = sap_gui_auto.GetScriptingEngine
    connection = application.Children(0)
    session = connection.Children(0)
    return sap_gui_auto, application, connection, session


def get_visible_row_count(grid):
    try:
        visible_count = int(grid.VisibleRowCount)
        if visible_count > 0:
            return visible_count
    except Exception:
        pass

    return 20


def scroll_sap_grid_to_row(grid, row_number):
    try:
        grid.firstVisibleRow = row_number
        return True
    except Exception:
        pass

    try:
        grid.VerticalScrollbar.Position = row_number
        return True
    except Exception:
        pass

    return False


def safe_get_sap_cell(grid, row_number, column_name):
    for attempt in range(3):
        try:
            return grid.GetCellValue(row_number, column_name)
        except Exception:
            scroll_sap_grid_to_row(grid, row_number)
            time.sleep(0.2)

    return ""


# =========================================================
# Main Script
# =========================================================

excel_file_path = r"C:\Users\E10878194\OneDrive - RTX\SAP Script Testing\GTF SS Database - TEST.xlsx"

notification_button_id = "wnd[0]/usr/btn%_S29_%_APP_%-VALU_PUSH"

xl_up = -4162

# =========================================================
# Column Mapping
# =========================================================

notification_col = 1       # Column A
object_no_col = 3          # Column C
disp_mat_col = 4           # Column D
disp_qty_col = 5           # Column E

xl_app = None
xl_book = None
xl_sheet = None
temp_book = None
temp_sheet = None

sap_gui_auto = None
application = None
connection = None
session = None
grid = None

sap_data_map = {}

try:
    # =========================================================
    # Open Excel and copy valid Notification Numbers from Column A
    # =========================================================

    if not os.path.exists(excel_file_path):
        msgbox(f"Excel file not found. Please check this path:\n{excel_file_path}")
        raise SystemExit

    xl_app = win32com.client.Dispatch("Excel.Application")
    xl_app.Visible = False
    xl_app.DisplayAlerts = False

    xl_book = xl_app.Workbooks.Open(excel_file_path, 0, False)

    if xl_book.ReadOnly:
        msgbox(
            "Excel file opened as Read-Only. Please check OneDrive/SharePoint sync permissions:\n"
            f"{excel_file_path}"
        )
        raise SystemExit

    xl_sheet = xl_book.Worksheets(1)

    last_row = xl_sheet.Cells(xl_sheet.Rows.Count, notification_col).End(xl_up).Row

    if last_row < 2:
        msgbox("No Notification Number found in Column A.")
        raise SystemExit

    temp_book = xl_app.Workbooks.Add()
    temp_sheet = temp_book.Worksheets(1)
    temp_sheet.Columns("A").NumberFormat = "@"

    temp_row = 1

    for i in range(2, last_row + 1):
        notif_no = clean_numeric_for_sap(get_cell_text(xl_sheet.Cells(i, notification_col)))

        if notif_no != "" and notif_no.upper() != "NOT FOUND" and notif_no.upper() != "NOTFOUND":
            temp_sheet.Cells(temp_row, 1).Value = notif_no
            temp_row += 1

    if temp_row == 1:
        msgbox("No valid Notification Number found in Column A.")
        raise SystemExit

    temp_sheet.Range(f"A1:A{temp_row - 1}").Copy()

    # =========================================================
    # Connect to SAP
    # =========================================================

    sap_gui_auto, application, connection, session = get_sap_session()

    # =========================================================
    # Open ZTBV and load Z50CFG_ENG_CRNT
    # =========================================================

    session.findById("wnd[0]").maximize()

    session.findById("wnd[0]/tbar[0]/okcd").Text = "/nZTBV"
    session.findById("wnd[0]").sendVKey(0)

    time.sleep(2)

    session.findById("wnd[0]/usr/txtD_WERKS").Text = "ESA1"
    session.findById("wnd[0]/usr/ctxtD_TAB").Text = "Z50CFG_ENG_CRNT"
    session.findById("wnd[0]/usr/ctxtD_TAB").SetFocus()
    session.findById("wnd[0]/usr/ctxtD_TAB").caretPosition = 15

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(2)

    # =========================================================
    # Paste Notification Numbers into S29
    # =========================================================

    session.findById(notification_button_id).press()

    time.sleep(1)

    # Added: clear values kept from a previous run -- SAP retains the
    # multi-select dialog's contents within a session, and leftovers would
    # silently join this run's filter. Button 16 = Delete Entire Selection
    # (absent on some SAP GUI versions, hence the try).
    try:
        session.findById("wnd[1]/tbar[0]/btn[16]").press()
        time.sleep(0.5)
    except Exception:
        pass

    session.findById("wnd[1]/tbar[0]/btn[24]").press()

    time.sleep(1)

    session.findById("wnd[1]/tbar[0]/btn[8]").press()

    time.sleep(1)

    xl_app.CutCopyMode = False

    # =========================================================
    # Execute Search
    # =========================================================

    session.findById("wnd[0]").sendVKey(8)

    time.sleep(3)

    if temp_book is not None:
        temp_book.Close(False)
        temp_book = None

    # =========================================================
    # Extract SAP data and match by Notification Number
    # =========================================================

    grid = session.findById("wnd[0]/shellcont/shell")

    row_count = int(grid.RowCount)
    visible_row_count = get_visible_row_count(grid)

    failed_read_count = 0

    for start_row in range(0, row_count, visible_row_count):
        scroll_sap_grid_to_row(grid, start_row)
        time.sleep(0.3)

        end_row = min(start_row + visible_row_count, row_count)

        for sap_row in range(start_row, end_row):
            try:
                scroll_sap_grid_to_row(grid, sap_row)

                sap_notif_no = normalize_key(safe_get_sap_cell(grid, sap_row, "QMNUM"))

                if sap_notif_no != "" and sap_notif_no not in sap_data_map:
                    sap_data_map[sap_notif_no] = {
                        "OBJNR": safe_get_sap_cell(grid, sap_row, "OBJNR"),
                        "DISP_MATNR": safe_get_sap_cell(grid, sap_row, "DISP_MATNR"),
                        "DISP_QTY": safe_get_sap_cell(grid, sap_row, "DISP_QTY"),
                    }

            except Exception:
                failed_read_count += 1

    # =========================================================
    # Paste matched results into Excel Columns C to E
    # Count matched and unmatched rows
    # =========================================================

    xl_sheet.Range(f"C2:E{xl_sheet.Rows.Count}").ClearContents()
    xl_sheet.Range("C:E").NumberFormat = "@"

    matched_count = 0
    not_matched_count = 0
    blank_count = 0

    for i in range(2, last_row + 1):
        excel_notif_no = normalize_key(get_cell_text(xl_sheet.Cells(i, notification_col)))  # Column A

        if excel_notif_no == "":
            blank_count += 1

            xl_sheet.Cells(i, object_no_col).Value = ""
            xl_sheet.Cells(i, disp_mat_col).Value = ""
            xl_sheet.Cells(i, disp_qty_col).Value = ""

        elif excel_notif_no in sap_data_map:
            xl_sheet.Cells(i, object_no_col).Value = sap_data_map[excel_notif_no]["OBJNR"]       # Column C
            xl_sheet.Cells(i, disp_mat_col).Value = sap_data_map[excel_notif_no]["DISP_MATNR"]   # Column D
            xl_sheet.Cells(i, disp_qty_col).Value = sap_data_map[excel_notif_no]["DISP_QTY"]     # Column E

            matched_count += 1

        else:
            xl_sheet.Cells(i, object_no_col).Value = ""
            xl_sheet.Cells(i, disp_mat_col).Value = ""
            xl_sheet.Cells(i, disp_qty_col).Value = ""

            not_matched_count += 1

    xl_sheet.Columns("C:E").AutoFit()

    xl_book.Save()

    msgbox(
        "Completed.\n\n"
        f"SAP Z50CFG_ENG_CRNT rows detected: {row_count}\n"
        f"SAP visible rows per screen: {visible_row_count}\n"
        f"Failed SAP row reads: {failed_read_count}\n\n"
        f"Matched Notification Numbers: {matched_count}\n"
        f"Unmatched Notification Numbers: {not_matched_count}\n"
        f"Blank Notification rows skipped: {blank_count}\n\n"
        "Notification Numbers were read from Column A.\n"
        "Object Number, Disposition Material, and Disp Qty have been pasted into Columns C, D, and E."
    )

except SystemExit:
    pass

except Exception as e:
    msgbox(f"Error occurred:\n{str(e)}")

finally:
    try:
        if temp_book is not None:
            temp_book.Close(False)
    except Exception:
        pass

    try:
        if xl_book is not None:
            xl_book.Close(True)
    except Exception:
        pass

    try:
        if xl_app is not None:
            xl_app.Quit()
    except Exception:
        pass

    sap_data_map = None
    grid = None
    temp_sheet = None
    temp_book = None
    xl_sheet = None
    xl_book = None
    xl_app = None
    session = None
    connection = None
    application = None
    sap_gui_auto = None

## Step 2: Z50CFG_ENG_VALD

In [ ]:
import os
import time
import ctypes
import win32com.client


# =========================================================
# Helper Functions
# =========================================================

def msgbox(message, title="SAP Python Script"):
    ctypes.windll.user32.MessageBoxW(0, message, title, 0)


def normalize_key(value):
    if value is None:
        txt = ""
    else:
        txt = str(value).strip()

    txt = txt.replace(" ", "")
    txt = txt.replace("\t", "")
    txt = txt.replace(chr(160), "")

    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]

    return txt


def get_cell_text(cell):
    try:
        txt = str(cell.Text).strip()
    except Exception:
        txt = ""

    if txt == "" or "#" in txt:
        try:
            txt = str(cell.Value).strip()
        except Exception:
            txt = ""

    return txt


def get_sap_session():
    sap_gui_auto = win32com.client.GetObject("SAPGUI")
    application = sap_gui_auto.GetScriptingEngine
    connection = application.Children(0)
    session = connection.Children(0)
    return sap_gui_auto, application, connection, session


def get_visible_row_count(grid):
    try:
        visible_count = int(grid.VisibleRowCount)
        if visible_count > 0:
            return visible_count
    except Exception:
        pass

    return 20


def scroll_sap_grid_to_row(grid, row_number):
    try:
        grid.firstVisibleRow = row_number
        return True
    except Exception:
        pass

    try:
        grid.VerticalScrollbar.Position = row_number
        return True
    except Exception:
        pass

    return False


def safe_get_sap_cell(grid, row_number, column_name):
    for attempt in range(3):
        try:
            return grid.GetCellValue(row_number, column_name)
        except Exception:
            scroll_sap_grid_to_row(grid, row_number)
            time.sleep(0.2)

    return ""


# =========================================================
# Main Script
# =========================================================

excel_file_path = r"C:\Users\E10878194\OneDrive - RTX\SAP Script Testing\GTF SS Database - TEST.xlsx"

xl_up = -4162

# =========================================================
# Column Mapping After Shifting One Column Right
# =========================================================

object_no_col = 3       # Column C

section_col = 6         # Column F
module_col = 7          # Column G
description_col = 8     # Column H
sales_doc_col = 9       # Column I
lid_col = 10            # Column J

xl_app = None
xl_book = None
xl_sheet = None
temp_book = None
temp_sheet = None

sap_gui_auto = None
application = None
connection = None
session = None
grid = None

sap_data_map = {}

try:
    # =========================================================
    # Open Excel and copy valid Object Numbers from Column C
    # =========================================================

    if not os.path.exists(excel_file_path):
        msgbox(f"Excel file not found. Please check this path:\n{excel_file_path}")
        raise SystemExit

    xl_app = win32com.client.Dispatch("Excel.Application")
    xl_app.Visible = False
    xl_app.DisplayAlerts = False

    xl_book = xl_app.Workbooks.Open(excel_file_path, 0, False)

    if xl_book.ReadOnly:
        msgbox(
            "Excel file opened as Read-Only. Please check OneDrive/SharePoint sync permissions:\n"
            f"{excel_file_path}"
        )
        raise SystemExit

    xl_sheet = xl_book.Worksheets(1)

    last_row = xl_sheet.Cells(xl_sheet.Rows.Count, object_no_col).End(xl_up).Row

    if last_row < 2:
        msgbox("No Object Number found in Column C.")
        raise SystemExit

    temp_book = xl_app.Workbooks.Add()
    temp_sheet = temp_book.Worksheets(1)
    temp_sheet.Columns("A").NumberFormat = "@"

    temp_row = 1

    for i in range(2, last_row + 1):
        object_no = get_cell_text(xl_sheet.Cells(i, object_no_col))  # Column C

        if object_no != "" and object_no.upper() != "NOT FOUND" and object_no.upper() != "NOTFOUND":
            temp_sheet.Cells(temp_row, 1).Value = object_no
            temp_row += 1

    if temp_row == 1:
        msgbox("No valid Object Number found in Column C.")
        raise SystemExit

    temp_sheet.Range(f"A1:A{temp_row - 1}").Copy()

    # =========================================================
    # Connect to SAP
    # =========================================================

    sap_gui_auto, application, connection, session = get_sap_session()

    # =========================================================
    # Open ZTBV and load Z50CFG_ENG_VALD
    # =========================================================

    session.findById("wnd[0]").maximize()
    session.findById("wnd[0]/tbar[0]/okcd").Text = "/nZTBV"
    session.findById("wnd[0]").sendVKey(0)

    time.sleep(2)

    session.findById("wnd[0]/usr/txtD_WERKS").Text = "ESA1"
    session.findById("wnd[0]/usr/ctxtD_TAB").Text = "Z50CFG_ENG_VALD"
    session.findById("wnd[0]/usr/ctxtD_TAB").SetFocus()
    session.findById("wnd[0]/usr/ctxtD_TAB").caretPosition = 15

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(1)

    # =========================================================
    # Paste Object Numbers into S2 and execute
    # =========================================================

    session.findById("wnd[0]/usr/btn%_S2_%_APP_%-VALU_PUSH").press()

    time.sleep(1)

    # Added: clear values kept from a previous run -- SAP retains the
    # multi-select dialog's contents within a session, and leftovers would
    # silently join this run's filter. Button 16 = Delete Entire Selection
    # (absent on some SAP GUI versions, hence the try).
    try:
        session.findById("wnd[1]/tbar[0]/btn[16]").press()
        time.sleep(0.5)
    except Exception:
        pass

    session.findById("wnd[1]/tbar[0]/btn[24]").press()

    time.sleep(1)

    session.findById("wnd[1]/tbar[0]/btn[8]").press()

    time.sleep(1)

    session.findById("wnd[0]/tbar[1]/btn[8]").press()

    time.sleep(3)

    xl_app.CutCopyMode = False

    if temp_book is not None:
        temp_book.Close(False)
        temp_book = None

    # =========================================================
    # Extract SAP data and match by Object Number
    # =========================================================

    grid = session.findById("wnd[0]/shellcont/shell")

    row_count = int(grid.RowCount)
    visible_row_count = get_visible_row_count(grid)

    failed_read_count = 0

    for start_row in range(0, row_count, visible_row_count):
        scroll_sap_grid_to_row(grid, start_row)
        time.sleep(0.3)

        end_row = min(start_row + visible_row_count, row_count)

        for sap_row in range(start_row, end_row):
            try:
                scroll_sap_grid_to_row(grid, sap_row)

                sap_obj_no = normalize_key(safe_get_sap_cell(grid, sap_row, "OBJNR"))

                if sap_obj_no != "" and sap_obj_no not in sap_data_map:
                    sap_data_map[sap_obj_no] = {
                        "Z_SECTION": safe_get_sap_cell(grid, sap_row, "Z_SECTION"),
                        "Z_MODULE": safe_get_sap_cell(grid, sap_row, "Z_MODULE"),
                        "DESCRIPT": safe_get_sap_cell(grid, sap_row, "DESCRIPT"),
                        "SALES_ORDER": safe_get_sap_cell(grid, sap_row, "SALES_ORDER"),
                        "LID": safe_get_sap_cell(grid, sap_row, "LID"),
                    }

            except Exception:
                failed_read_count += 1

    # =========================================================
    # Paste matched results into Excel Columns F to I
    # =========================================================

    xl_sheet.Range(f"F2:I{xl_sheet.Rows.Count}").ClearContents()
    xl_sheet.Range("F:I").NumberFormat = "@"

    matched_count = 0
    not_matched_count = 0
    blank_count = 0

    for i in range(2, last_row + 1):
        excel_obj_no = normalize_key(get_cell_text(xl_sheet.Cells(i, object_no_col)))  # Column C

        if excel_obj_no == "":
            blank_count += 1

            xl_sheet.Cells(i, section_col).Value = ""
            xl_sheet.Cells(i, module_col).Value = ""
            xl_sheet.Cells(i, description_col).Value = ""
            xl_sheet.Cells(i, sales_doc_col).Value = ""
            xl_sheet.Cells(i, lid_col).Value = ""

        elif excel_obj_no in sap_data_map:
            xl_sheet.Cells(i, section_col).Value = sap_data_map[excel_obj_no]["Z_SECTION"]          # Column F
            xl_sheet.Cells(i, module_col).Value = sap_data_map[excel_obj_no]["Z_MODULE"]            # Column G
            xl_sheet.Cells(i, description_col).Value = sap_data_map[excel_obj_no]["DESCRIPT"]       # Column H
            xl_sheet.Cells(i, sales_doc_col).Value = sap_data_map[excel_obj_no]["SALES_ORDER"]      # Column I

            matched_count += 1

        else:
            xl_sheet.Cells(i, section_col).Value = ""
            xl_sheet.Cells(i, module_col).Value = ""
            xl_sheet.Cells(i, description_col).Value = ""
            xl_sheet.Cells(i, sales_doc_col).Value = ""
            xl_sheet.Cells(i, lid_col).Value = ""

            not_matched_count += 1

    xl_sheet.Columns("F:I").AutoFit()

    xl_book.Save()

    msgbox(
        "Completed.\n\n"
        f"SAP Z50CFG_ENG_VALD rows detected: {row_count}\n"
        f"SAP visible rows per screen: {visible_row_count}\n"
        f"Failed SAP row reads: {failed_read_count}\n\n"
        f"Matched Object Numbers: {matched_count}\n"
        f"Unmatched Object Numbers: {not_matched_count}\n"
        f"Blank Object Number rows skipped: {blank_count}\n\n"
        "Object Numbers were read from Column C.\n"
        "Section, Module, Description, Sales Doc., and LID have been pasted into Columns F to J."
    )

except SystemExit:
    pass

except Exception as e:
    msgbox(f"Error occurred:\n{str(e)}")

finally:
    try:
        if temp_book is not None:
            temp_book.Close(False)
    except Exception:
        pass

    try:
        if xl_book is not None:
            xl_book.Close(True)
    except Exception:
        pass

    try:
        if xl_app is not None:
            xl_app.Quit()
    except Exception:
        pass

    sap_data_map = None
    grid = None
    temp_sheet = None
    temp_book = None
    xl_sheet = None
    xl_book = None
    xl_app = None
    session = None
    connection = None
    application = None
    sap_gui_auto = None